In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
from pathlib import Path
import os
import re
from typing import Dict, List

import duckdb
import pandas as pd
from pymongo import MongoClient
from dotenv import load_dotenv
from google import genai

In [3]:
# Load environment variables

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found in environment variables or .env file.")

True

In [4]:
PROJECT_ROOT = Path.cwd()
DUCKDB_PATH = PROJECT_ROOT / "data" / "warehouse" / "project.duckdb"

REPORTS_DIR = PROJECT_ROOT / "artifacts" / "reports"
TABLES_DIR = PROJECT_ROOT / "artifacts" / "tables"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Connect DuckDB
duckdb_con = duckdb.connect(str(DUCKDB_PATH))

# Connect MongoDB
mongodb_client = MongoClient("mongodb://localhost:27017/")
mongodb = mongodb_client["uk_entity_review"]
policy_collection = mongodb["policy_documents"]

print("DuckDB connected:", DUCKDB_PATH)
print("MongoDB connected:", mongodb.name)

# Connect Gemini
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

print("Reports directory:", REPORTS_DIR)
print("DuckDB connected:", DUCKDB_PATH)
print("MongoDB connected:", mongodb.name)
print("Gemini key loaded:", GEMINI_API_KEY is not None)

DuckDB connected: /Users/lingzitong/Desktop/MSIN0166 Individual Assignment/data/warehouse/project.duckdb
MongoDB connected: uk_entity_review
Reports directory: /Users/lingzitong/Desktop/MSIN0166 Individual Assignment/artifacts/reports
DuckDB connected: /Users/lingzitong/Desktop/MSIN0166 Individual Assignment/data/warehouse/project.duckdb
MongoDB connected: uk_entity_review
Gemini key loaded: True


In [5]:
policy_docs = list(
    policy_collection.find(
        {},
        {"_id": 0, "doc_id": 1, "title": 1, "content": 1}
    )
)

print("Number of policy docs loaded:", len(policy_docs))
[doc["title"] for doc in policy_docs]

Number of policy docs loaded: 3


['Entity Review Policy', 'Risk Flag Guidelines', 'Manual Review Checklist']

In [6]:
def split_markdown_into_sections(title: str, content: str) -> List[Dict]:
    """
    Split a markdown document into section-level chunks using '## ' headings.
    If no '##' headings exist, keep the full document as one chunk.
    """
    if "## " not in content:
        return [{
            "source_title": title,
            "section_id": f"{title.lower().replace(' ', '_')}_001",
            "section_heading": "Full document",
            "content": content.strip()
        }]

    parts = re.split(r"(?=^##\s)", content, flags=re.MULTILINE)
    chunks = []

    chunk_counter = 1
    for part in parts:
        part = part.strip()
        if not part:
            continue

        lines = part.splitlines()
        heading_line = lines[0] if lines else f"Section {chunk_counter}"
        section_heading = heading_line.replace("##", "").strip()

        chunks.append({
            "source_title": title,
            "section_id": f"{title.lower().replace(' ', '_')}_{chunk_counter:03d}",
            "section_heading": section_heading,
            "content": part
        })
        chunk_counter += 1

    return chunks

In [7]:
policy_chunks = []

for doc in policy_docs:
    policy_chunks.extend(
        split_markdown_into_sections(doc["title"], doc["content"])
    )

policy_chunks_df = pd.DataFrame(policy_chunks)
policy_chunks_df.head(10)

,source_title,section_id,section_heading,content
0,Entity Review Policy,entity_review_policy_001,# Entity Review Policy,# Entity Review Policy
1,Entity Review Policy,entity_review_policy_002,Purpose,## Purpose\nThis document defines how the prot...
2,Entity Review Policy,entity_review_policy_003,Review Scope,## Review Scope\nThe prototype focuses on oper...
3,Entity Review Policy,entity_review_policy_004,Review Principles,## Review Principles\nThe review process follo...
4,Entity Review Policy,entity_review_policy_005,Priority Bands,## Priority Bands\n- **High**: The entity exhi...
5,Entity Review Policy,entity_review_policy_006,Prototype Limitation,## Prototype Limitation\nThis policy is design...
6,Risk Flag Guidelines,risk_flag_guidelines_001,# Risk Flag Guidelines,# Risk Flag Guidelines
7,Risk Flag Guidelines,risk_flag_guidelines_002,Purpose,## Purpose\nThis document explains the rule-ba...
8,Risk Flag Guidelines,risk_flag_guidelines_003,1. `new_entity_flag`,## 1. `new_entity_flag`\n### Definition\nTrigg...
9,Risk Flag Guidelines,risk_flag_guidelines_004,2. `missing_location_flag`,## 2. `missing_location_flag`\n### Definition\...


In [8]:
chunk_count_summary = (
    policy_chunks_df.groupby("source_title")
    .size()
    .reset_index(name="chunk_count")
    .sort_values("chunk_count", ascending=False)
)

chunk_count_summary

,source_title,chunk_count
1,Manual Review Checklist,9
2,Risk Flag Guidelines,8
0,Entity Review Policy,6


In [9]:
policy_chunks_output_path = TABLES_DIR / "policy_chunks_summary.csv"
policy_chunks_df.to_csv(policy_chunks_output_path, index=False)

print("Saved policy chunk table to:", policy_chunks_output_path)

Saved policy chunk table to: /Users/lingzitong/Desktop/MSIN0166 Individual Assignment/artifacts/tables/policy_chunks_summary.csv


In [10]:
def format_entity_brief(entity_record: Dict) -> str:
    lines = [
        f"Entity ID: {entity_record.get('entity_id')}",
        f"Entity Name: {entity_record.get('entity_name')}",
        f"Account Category: {entity_record.get('account_category')}",
        f"Primary SIC: {entity_record.get('sic_text_1')}",
        f"Incorporation Date: {str(entity_record.get('incorporation_date'))[:10]}",
        f"Mortgage Charges: {entity_record.get('num_mort_charges')}",
        f"Outstanding Mortgage Charges: {entity_record.get('num_mort_outstanding')}",
        f"Review Priority Score: {entity_record.get('review_priority_score')}",
        f"Review Priority Band: {entity_record.get('review_priority_band')}",
    ]
    return "\n".join(lines)


def get_triggered_signals(entity_record: Dict) -> List[str]:
    signal_columns = [
        "new_entity_flag",
        "missing_location_flag",
        "no_accounts_filed_flag",
        "has_outstanding_mortgage_flag",
        "mixed_mortgage_profile_flag",
    ]
    return [signal_name for signal_name in signal_columns if entity_record.get(signal_name) == 1]


def format_triggered_signals(triggered_signals: List[str]) -> str:
    signal_labels = {
        "new_entity_flag": "New entity signal",
        "missing_location_flag": "Missing location signal",
        "no_accounts_filed_flag": "No accounts filed signal",
        "has_outstanding_mortgage_flag": "Outstanding charge signal",
        "mixed_mortgage_profile_flag": "Mixed mortgage profile signal",
    }

    signal_explanations = {
        "new_entity_flag": "The entity is recently incorporated, so the amount of historical operating and filing information available for review is still limited.",
        "missing_location_flag": "The registry profile contains incomplete location information, which weakens basic profile completeness and traceability.",
        "no_accounts_filed_flag": "No filed accounts are currently visible in the registry profile, which reduces disclosure visibility and limits routine filing-based review.",
        "has_outstanding_mortgage_flag": "The entity has outstanding registered charges, which may justify closer review of its structural and financing profile.",
        "mixed_mortgage_profile_flag": "The entity shows a mixed mortgage profile, where outstanding charges coexist with satisfied or part-satisfied charge history, indicating a more involved charge structure.",
    }

    lines = []
    for signal in triggered_signals:
        label = signal_labels.get(signal, signal)
        explanation = signal_explanations.get(signal, "No explanation available.")
        lines.append(f"- {label}: {explanation}")
    return "\n".join(lines)


def format_enrichment_brief(enrichment_record: Dict) -> str:
    if not enrichment_record:
        return "No external API enrichment available."

    lines = [
        f"API Company Status: {enrichment_record.get('company_status_api')}",
        f"API Date of Creation: {enrichment_record.get('date_of_creation_api')}",
        f"Has Insolvency History: {enrichment_record.get('has_insolvency_history')}",
        f"API Type: {enrichment_record.get('type_api')}",
        f"Jurisdiction: {enrichment_record.get('jurisdiction')}",
    ]
    return "\n".join(lines)

In [11]:
signal_keyword_map = {
    "new_entity_flag": [
        "recently incorporated",
        "incorporated",
        "shareholder register",
        "operating history",
        "newly incorporated"
    ],
    "missing_location_flag": [
        "location",
        "address",
        "registered office",
        "traceability",
        "completeness"
    ],
    "no_accounts_filed_flag": [
        "accounts",
        "filed accounts",
        "disclosure",
        "filing",
        "report and accounts"
    ],
    "has_outstanding_mortgage_flag": [
        "charge",
        "registered charge",
        "mortgage",
        "secured",
        "outstanding"
    ],
    "mixed_mortgage_profile_flag": [
        "charge",
        "mortgage",
        "outstanding",
        "satisfied",
        "part-satisfied"
    ],
}

In [12]:
def retrieve_relevant_chunks(
    triggered_signals: List[str],
    chunks_df: pd.DataFrame,
    top_k: int = 5
) -> pd.DataFrame:
    keywords = []
    for signal in triggered_signals:
        keywords.extend(signal_keyword_map.get(signal, []))

    keywords = list(set(keywords))

    if not keywords:
        result_df = chunks_df.copy()
        result_df["retrieval_score"] = 0
        return result_df.head(top_k)

    def score_text(text: str) -> int:
        text_lower = text.lower()
        return sum(1 for kw in keywords if kw.lower() in text_lower)

    scored_df = chunks_df.copy()
    scored_df["retrieval_score"] = scored_df["content"].apply(score_text)

    ranked_df = (
        scored_df.sort_values(
            by=["retrieval_score", "source_title", "section_heading"],
            ascending=[False, True, True]
        )
    )

    filtered_df = ranked_df[ranked_df["retrieval_score"] > 0]

    if filtered_df.empty:
        return ranked_df.head(top_k)

    return filtered_df.head(top_k)

In [13]:
def format_rag_policy_snippets(retrieved_chunks_df: pd.DataFrame, max_chars_per_chunk: int = 1000) -> str:
    snippets = []

    for _, row in retrieved_chunks_df.iterrows():
        snippet_text = row["content"][:max_chars_per_chunk]
        snippets.append(
            f"{row['source_title']} — {row['section_heading']}:\n{snippet_text}"
        )

    return "\n\n".join(snippets)

In [14]:
def generate_llm_review_note(review_prompt: str) -> str:
    response = gemini_client.models.generate_content(
        model="gemini-3.1-flash-lite-preview",
        contents=review_prompt,
    )
    return response.text

In [15]:
def generate_rule_based_review_note(
    entity_record: Dict,
    enrichment_record: Dict,
    triggered_signals: List[str]
) -> str:
    entity_name = entity_record.get("entity_name", "Unknown Entity")
    priority_band = entity_record.get("review_priority_band", "Unknown")
    review_score = entity_record.get("review_priority_score", "Unknown")

    rationale_line = (
        f"{entity_name} is classified as {priority_band} priority with a review score of {review_score}."
    )

    signal_text = format_triggered_signals(triggered_signals)

    if enrichment_record:
        enrichment_text = (
            f"The Companies House API indicates company status = {enrichment_record.get('company_status_api')}, "
            f"type = {enrichment_record.get('type_api')}, jurisdiction = {enrichment_record.get('jurisdiction')}, "
            f"and insolvency history = {enrichment_record.get('has_insolvency_history')}."
        )
    else:
        enrichment_text = "No external API enrichment was available for this entity."

    review_considerations = [
        "- Assess whether the absence of filed accounts is consistent with the incorporation date and the expected statutory filing schedule.",
        "- Conduct a more detailed review of the registered charge profile, with particular attention to the volume and legal status of active charges.",
        "- Assess whether the ownership and control structure appears sufficiently clear or whether further verification is required.",
        "- Consider whether the current review profile may warrant further documentary review or enhanced review.",
    ]

    review_note = f"""
Risk Review Support Note — {entity_name}

1. Review priority rationale
{rationale_line}

2. Triggered indicators
{signal_text}

3. External official context
{enrichment_text}

4. Further review considerations
{"\n".join(review_considerations)}
""".strip()

    return review_note

In [16]:
high_entities_df = duckdb_con.execute("""
    SELECT
        entity_id,
        entity_name,
        incorporation_date,
        account_category,
        num_mort_charges,
        num_mort_outstanding,
        sic_text_1,
        new_entity_flag,
        missing_location_flag,
        no_accounts_filed_flag,
        has_outstanding_mortgage_flag,
        mixed_mortgage_profile_flag,
        review_priority_score,
        review_priority_band
    FROM entity_risk_signals_v2
    WHERE review_priority_band = 'High'
    ORDER BY entity_id
    LIMIT 2
""").fetchdf()

medium_entities_df = duckdb_con.execute("""
    SELECT
        entity_id,
        entity_name,
        incorporation_date,
        account_category,
        num_mort_charges,
        num_mort_outstanding,
        sic_text_1,
        new_entity_flag,
        missing_location_flag,
        no_accounts_filed_flag,
        has_outstanding_mortgage_flag,
        mixed_mortgage_profile_flag,
        review_priority_score,
        review_priority_band
    FROM entity_risk_signals_v2
    WHERE review_priority_band = 'Medium'
    ORDER BY entity_id
    LIMIT 1
""").fetchdf()

sample_entities_df = pd.concat([high_entities_df, medium_entities_df], ignore_index=True)
sample_entities_df

,entity_id,entity_name,incorporation_date,account_category,num_mort_charges,num_mort_outstanding,sic_text_1,new_entity_flag,missing_location_flag,no_accounts_filed_flag,has_outstanding_mortgage_flag,mixed_mortgage_profile_flag,review_priority_score,review_priority_band
0,16332344,BLUBRIGHT PROPERTY LTD,2025-03-21,NO ACCOUNTS FILED,2,1,68209 - Other letting and operating of own or ...,1,0,1,1,1,4,High
1,16333879,OVERBROOK HOLDINGS LIMITED,2025-03-21,NO ACCOUNTS FILED,3,2,41100 - Development of building projects,1,0,1,1,1,4,High
2,00046050,COOPER BROTHERS & SONS LIMITED,1895-11-22,NO ACCOUNTS FILED,2,1,None Supplied,0,0,1,1,1,3,Medium


In [17]:
def build_rag_review_package_for_entity(entity_row: pd.Series) -> Dict:
    entity_record = entity_row.to_dict()
    entity_id = entity_record["entity_id"]

    external_df = duckdb_con.execute(f"""
        SELECT *
        FROM entity_external_enrichment
        WHERE entity_id = '{entity_id}'
    """).fetchdf()

    if not external_df.empty:
        enrichment_record = external_df.iloc[0].to_dict()
    else:
        enrichment_record = {}

    triggered_signals = get_triggered_signals(entity_record)
    retrieved_chunks_df = retrieve_relevant_chunks(
        triggered_signals=triggered_signals,
        chunks_df=policy_chunks_df,
        top_k=5
    )

    entity_brief = format_entity_brief(entity_record)
    triggered_signals_text = format_triggered_signals(triggered_signals)
    enrichment_brief = format_enrichment_brief(enrichment_record)
    rag_policy_text = format_rag_policy_snippets(retrieved_chunks_df, max_chars_per_chunk=1000)

    review_prompt = f"""
You are assisting with a UK business entity risk review support workflow.

Your role is to produce a structured review support note for a human analyst.
Do not make final legal, regulatory, or compliance determinations.
Do not state that misconduct, illegality, or sanctions exposure has been established.

Entity summary:
{entity_brief}

Triggered review indicators:
{triggered_signals_text}

External official enrichment:
{enrichment_brief}

Retrieved policy guidance snippets:
{rag_policy_text}

Write a concise professional review support note with the following headings:
1. Review priority rationale
2. Triggered indicators
3. External official context
4. Further review considerations

Keep the tone analytical, cautious, and supportive of human review.
""".strip()

    return {
        "entity_record": entity_record,
        "enrichment_record": enrichment_record,
        "triggered_signals": triggered_signals,
        "retrieved_chunks_df": retrieved_chunks_df,
        "entity_brief": entity_brief,
        "enrichment_brief": enrichment_brief,
        "rag_policy_text": rag_policy_text,
        "review_prompt": review_prompt,
    }

In [18]:
sample_rag_package = build_rag_review_package_for_entity(sample_entities_df.iloc[0])

print(sample_rag_package["entity_brief"])
print("\n--- Triggered signals ---")
print(format_triggered_signals(sample_rag_package["triggered_signals"]))
print("\n--- Retrieved chunks ---")
display(sample_rag_package["retrieved_chunks_df"][["source_title", "section_heading", "retrieval_score"]])
print("\n--- Prompt preview ---")
print(sample_rag_package["review_prompt"][:5000])

Entity ID: 16332344
Entity Name: BLUBRIGHT PROPERTY LTD
Account Category: NO ACCOUNTS FILED
Primary SIC: 68209 - Other letting and operating of own or leased real estate
Incorporation Date: 2025-03-21
Mortgage Charges: 2
Outstanding Mortgage Charges: 1
Review Priority Score: 4
Review Priority Band: High

--- Triggered signals ---
- New entity signal: The entity is recently incorporated, so the amount of historical operating and filing information available for review is still limited.
- No accounts filed signal: No filed accounts are currently visible in the registry profile, which reduces disclosure visibility and limits routine filing-based review.
- Outstanding charge signal: The entity has outstanding registered charges, which may justify closer review of its structural and financing profile.
- Mixed mortgage profile signal: The entity shows a mixed mortgage profile, where outstanding charges coexist with satisfied or part-satisfied charge history, indicating a more involved charge

,source_title,section_heading,retrieval_score
3,Entity Review Policy,Review Principles,7
12,Risk Flag Guidelines,5. `mixed_mortgage_profile_flag`,7
19,Manual Review Checklist,Step 4: Review Charge / Mortgage Structure,6
8,Risk Flag Guidelines,1. `new_entity_flag`,5
11,Risk Flag Guidelines,4. `has_outstanding_mortgage_flag`,5



--- Prompt preview ---
You are assisting with a UK business entity risk review support workflow.

Your role is to produce a structured review support note for a human analyst.
Do not make final legal, regulatory, or compliance determinations.
Do not state that misconduct, illegality, or sanctions exposure has been established.

Entity summary:
Entity ID: 16332344
Entity Name: BLUBRIGHT PROPERTY LTD
Account Category: NO ACCOUNTS FILED
Primary SIC: 68209 - Other letting and operating of own or leased real estate
Incorporation Date: 2025-03-21
Mortgage Charges: 2
Outstanding Mortgage Charges: 1
Review Priority Score: 4
Review Priority Band: High

Triggered review indicators:
- New entity signal: The entity is recently incorporated, so the amount of historical operating and filing information available for review is still limited.
- No accounts filed signal: No filed accounts are currently visible in the registry profile, which reduces disclosure visibility and limits routine filing-based

In [19]:
sample_rag_prompt = sample_rag_package["review_prompt"]

try:
    sample_rag_llm_review_note = generate_llm_review_note(sample_rag_prompt)
except Exception as error:
    sample_rag_llm_review_note = None
    print("LLM call failed:", error)

sample_rag_rule_based_note = generate_rule_based_review_note(
    entity_record=sample_rag_package["entity_record"],
    enrichment_record=sample_rag_package["enrichment_record"],
    triggered_signals=sample_rag_package["triggered_signals"]
)

print("=== RAG-ENHANCED LLM REVIEW NOTE ===")
print(sample_rag_llm_review_note if sample_rag_llm_review_note else "No LLM note generated.")

print("\n\n=== RULE-BASED FALLBACK NOTE ===")
print(sample_rag_rule_based_note)

=== RAG-ENHANCED LLM REVIEW NOTE ===
### Review Support Note: BLUBRIGHT PROPERTY LTD (16332344)

**1. Review priority rationale**
This entity is flagged as High Priority (Score: 4) due to a combination of its very recent incorporation and a complex financing structure. As the entity is less than one month old, it lacks the historical filing record typical of established trading entities. The presence of outstanding mortgage charges alongside a mixed charge history suggests a structured financing arrangement that warrants verification alongside the entity's limited operational disclosures.

**2. Triggered indicators**
*   **New entity signal:** Incorporated 2025-03-21 (within the last 365 days), resulting in limited historical operating and filing data.
*   **No accounts filed signal:** No historical accounts are available for review, limiting visibility into financial performance and disclosure depth.
*   **Outstanding charge signal:** Confirmed existence of active secured obligations 

In [20]:
rag_review_outputs = []
retrieved_chunk_rows = []

for _, row in sample_entities_df.iterrows():
    rag_package = build_rag_review_package_for_entity(row)

    try:
        llm_review_note = generate_llm_review_note(rag_package["review_prompt"])
        llm_generation_status = "success"
    except Exception as error:
        llm_review_note = None
        llm_generation_status = f"failed: {error}"

    rule_based_review_note = generate_rule_based_review_note(
        entity_record=rag_package["entity_record"],
        enrichment_record=rag_package["enrichment_record"],
        triggered_signals=rag_package["triggered_signals"]
    )

    final_review_note = llm_review_note if llm_review_note else rule_based_review_note

    rag_review_outputs.append({
        "entity_id": rag_package["entity_record"]["entity_id"],
        "entity_name": rag_package["entity_record"]["entity_name"],
        "review_priority_band": rag_package["entity_record"]["review_priority_band"],
        "review_priority_score": rag_package["entity_record"]["review_priority_score"],
        "triggered_signals": rag_package["triggered_signals"],
        "llm_generation_status": llm_generation_status,
        "review_prompt": rag_package["review_prompt"],
        "review_note": final_review_note,
    })

    for _, chunk_row in rag_package["retrieved_chunks_df"].iterrows():
        retrieved_chunk_rows.append({
            "entity_id": rag_package["entity_record"]["entity_id"],
            "entity_name": rag_package["entity_record"]["entity_name"],
            "review_priority_band": rag_package["entity_record"]["review_priority_band"],
            "source_title": chunk_row["source_title"],
            "section_id": chunk_row["section_id"],
            "section_heading": chunk_row["section_heading"],
            "retrieval_score": chunk_row["retrieval_score"],
        })

rag_review_outputs_df = pd.DataFrame(rag_review_outputs)
retrieved_chunks_summary_df = pd.DataFrame(retrieved_chunk_rows)

rag_review_outputs_df[["entity_id", "entity_name", "review_priority_band", "review_priority_score", "llm_generation_status"]]

,entity_id,entity_name,review_priority_band,review_priority_score,llm_generation_status
0,16332344,BLUBRIGHT PROPERTY LTD,High,4,success
1,16333879,OVERBROOK HOLDINGS LIMITED,High,4,success
2,00046050,COOPER BROTHERS & SONS LIMITED,Medium,3,success


In [21]:
retrieved_chunks_summary_df

,entity_id,entity_name,review_priority_band,source_title,section_id,section_heading,retrieval_score
0,16332344,BLUBRIGHT PROPERTY LTD,High,Entity Review Policy,entity_review_policy_004,Review Principles,7
1,16332344,BLUBRIGHT PROPERTY LTD,High,Risk Flag Guidelines,risk_flag_guidelines_007,5. `mixed_mortgage_profile_flag`,7
2,16332344,BLUBRIGHT PROPERTY LTD,High,Manual Review Checklist,manual_review_checklist_006,Step 4: Review Charge / Mortgage Structure,6
3,16332344,BLUBRIGHT PROPERTY LTD,High,Risk Flag Guidelines,risk_flag_guidelines_003,1. `new_entity_flag`,5
4,16332344,BLUBRIGHT PROPERTY LTD,High,Risk Flag Guidelines,risk_flag_guidelines_006,4. `has_outstanding_mortgage_flag`,5
5,16333879,OVERBROOK HOLDINGS LIMITED,High,Entity Review Policy,entity_review_policy_004,Review Principles,7
6,16333879,OVERBROOK HOLDINGS LIMITED,High,Risk Flag Guidelines,risk_flag_guidelines_007,5. `mixed_mortgage_profile_flag`,7
7,16333879,OVERBROOK HOLDINGS LIMITED,High,Manual Review Checklist,manual_review_checklist_006,Step 4: Review Charge / Mortgage Structure,6
8,16333879,OVERBROOK HOLDINGS LIMITED,High,Risk Flag Guidelines,risk_flag_guidelines_003,1. `new_entity_flag`,5
9,16333879,OVERBROOK HOLDINGS LIMITED,High,Risk Flag Guidelines,risk_flag_guidelines_006,4. `has_outstanding_mortgage_flag`,5


In [22]:
for output in rag_review_outputs:
    entity_id = output["entity_id"]
    priority_band = str(output["review_priority_band"]).lower()

    note_path = REPORTS_DIR / f"rag_review_note_{priority_band}_{entity_id}.md"
    prompt_path = REPORTS_DIR / f"rag_review_prompt_{priority_band}_{entity_id}.txt"

    note_path.write_text(output["review_note"], encoding="utf-8")
    prompt_path.write_text(output["review_prompt"], encoding="utf-8")

print("Saved RAG review notes and prompts to:", REPORTS_DIR)

2278

4745

2553

4725

2295

4420

Saved RAG review notes and prompts to: /Users/lingzitong/Desktop/MSIN0166 Individual Assignment/artifacts/reports


In [23]:
rag_outputs_summary_path = TABLES_DIR / "rag_review_outputs_summary.csv"
retrieved_chunks_summary_path = TABLES_DIR / "retrieved_chunks_summary.csv"

rag_review_outputs_df.drop(columns=["review_prompt", "review_note"]).to_csv(rag_outputs_summary_path, index=False)
retrieved_chunks_summary_df.to_csv(retrieved_chunks_summary_path, index=False)

print("Saved RAG outputs summary to:", rag_outputs_summary_path)
print("Saved retrieved chunks summary to:", retrieved_chunks_summary_path)

Saved RAG outputs summary to: /Users/lingzitong/Desktop/MSIN0166 Individual Assignment/artifacts/tables/rag_review_outputs_summary.csv
Saved retrieved chunks summary to: /Users/lingzitong/Desktop/MSIN0166 Individual Assignment/artifacts/tables/retrieved_chunks_summary.csv


In [24]:
example_rag_note_path = REPORTS_DIR / f"rag_review_note_{str(rag_review_outputs[0]['review_priority_band']).lower()}_{rag_review_outputs[0]['entity_id']}.md"
print(example_rag_note_path.read_text(encoding="utf-8")[:4000])

### Entity Review Support Note: BLUBRIGHT PROPERTY LTD (16332344)

**1. Review priority rationale**
The entity is classified as High Priority (Score 4) primarily due to its status as a newly incorporated company combined with an active secured-debt profile. The absence of historical filing data, coupled with a complex charge structure, necessitates a cautious approach to establishing the entity’s operational and financial context.

**2. Triggered indicators**
*   **New entity signal:** Incorporated on 2025-03-21, providing no historical operating data or previous annual filings.
*   **No accounts filed signal:** Consistent with the recent incorporation date, no financial statements are available, limiting the ability to assess financial performance or trading activity.
*   **Outstanding charge signal:** The entity maintains active registered charges, indicating the presence of secured financial obligations.
*   **Mixed mortgage profile signal:** The existence of both outstanding and sa

In [26]:
duckdb_con.close()
mongodb_client.close()

print("DuckDB connection closed.")
print("MongoDB connection closed.")

DuckDB connection closed.
MongoDB connection closed.
